In [3]:
from spinnaker2 import snn, hardware
import numpy as np

In [4]:
import numpy as np
from spinnaker2 import snn, brian2_sim

# ------------------------------------------------------------
# Network
# ------------------------------------------------------------
net = snn.Network("chain_reaction")

# ------------------------------------------------------------
# 1. Stimulus: single spike at time step 10
# ------------------------------------------------------------
pop_input = snn.Population(
    1,
    "spike_list",
    {0: [10]},
    name="stimulus"
)

# ------------------------------------------------------------
# 2. Neuron parameters (no leak, easy spiking)
# ------------------------------------------------------------
params = {
    "v_threshold": 1.0,
    "alpha": 1.0,     # no decay
    "v_reset": 0.0,
    "v_rest": 0.0
}

layer_1 = snn.Population(
    1,
    "lif",
    params,
    name="layer_1",
    record=["spikes"]
)

layer_2 = snn.Population(
    1,
    "lif",
    params,
    name="layer_2",
    record=["spikes"]
)

# ------------------------------------------------------------
# 3. Connections: [pre, post, weight, delay]
# ------------------------------------------------------------

# Input -> Layer 1
conns_input = np.array([
    [0, 0, 5.0, 1]   # strong weight, short delay
])

proj_input = snn.Projection(pop_input, layer_1, conns_input)

# Layer 1 -> Layer 2
conns_internal = np.array([
    [0, 0, 5.0, 5]   # strong weight, visible propagation delay
])

proj_internal = snn.Projection(layer_1, layer_2, conns_internal)

# ------------------------------------------------------------
# 4. Build network
# ------------------------------------------------------------
net.add(
    pop_input,
    layer_1,
    layer_2,
    proj_input,
    proj_internal
)

# ------------------------------------------------------------
# 5. Run emulation
# ------------------------------------------------------------
print("Initializing Brian2 Backend...")
hw = brian2_sim.Brian2Backend()

print("Running simulation...")
hw.run(net, time_steps=100)

# ------------------------------------------------------------
# 6. Results
# ------------------------------------------------------------
spikes_1 = layer_1.get_spikes()
spikes_2 = layer_2.get_spikes()

print(f"Layer 1 Spikes: {spikes_1}")
print(f"Layer 2 Spikes: {spikes_2}")


Initializing Brian2 Backend...
Running simulation...
Starting simulation at t=0. s for a duration of 100. ms
100. ms (100%) simulated in < 1s
finished run
getting spikes from layer_1
getting spikes from layer_2
Layer 1 Spikes: {0: [12, 13, 14]}
Layer 2 Spikes: {0: [18, 19, 20, 21, 22, 23, 24]}
